# 06 · Counterfactual Explanations

In [ ]:
!pip install shap lime matplotlib seaborn pandas numpy scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.datasets import load_breast_cancer
import shap
import lime
import lime.lime_tabular

np.random.seed(42)


In [ ]:
# Load the Breast Cancer Wisconsin dataset (used throughout this course)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Train a Random Forest — a strong but harder-to-interpret "black box" model
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")


## Generate a Minimal-Change Counterfactual

In [ ]:
def generate_counterfactual(model, instance, feature_names, target_class=1, max_iter=1000):
    """
    Simple counterfactual generator.
    Tries to find minimal changes to flip prediction.
    """
    original_pred = model.predict([instance])[0]

    if original_pred == target_class:
        print(f"Instance already predicts target class {target_class}")
        return instance

    best_cf = None
    best_distance = float('inf')

    for _ in range(max_iter):
        noise = np.random.normal(0, 0.1, instance.shape)
        candidate = instance + noise
        candidate = np.clip(candidate, 0, instance.max() * 1.5)

        pred = model.predict([candidate])[0]

        if pred == target_class:
            distance = np.linalg.norm(candidate - instance)
            if distance < best_distance:
                best_distance = distance
                best_cf = candidate

    return best_cf

# Generate counterfactual for a malignant prediction
malignant_idx = np.where(y_pred_rf == 0)[0][0]
instance = X_test.iloc[malignant_idx].values

print(f"Original prediction: {'Malignant' if y_pred_rf[malignant_idx]==0 else 'Benign'}")
print("\nOriginal feature values:")
for i, (name, value) in enumerate(zip(data.feature_names, instance)):
    print(f"  {name}: {value:.3f}")

cf = generate_counterfactual(rf, instance, data.feature_names, target_class=1)

if cf is not None:
    print("\n--- COUNTERFACTUAL ---")
    print("Changes needed to flip prediction to Benign:")
    changes = []
    for i, (name, orig, new) in enumerate(zip(data.feature_names, instance, cf)):
        if abs(orig - new) > 0.01 * abs(orig):
            changes.append((name, orig, new))

    changes_df = pd.DataFrame(changes, columns=['Feature', 'Original', 'Counterfactual'])
    changes_df['Change'] = changes_df['Counterfactual'] - changes_df['Original']
    changes_df['Change %'] = (changes_df['Change'] / changes_df['Original']) * 100
    print(changes_df.to_string(index=False))
else:
    print("Could not find counterfactual")
